In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
import datetime
import joblib
from sklearn.model_selection import GridSearchCV
import os

In [12]:
# 1. LOAD DỮ LIỆU
path = '../../../data/insurance.csv'
if os.path.exists(path):
    print("✅ Đường dẫn ĐÚNG! File đã tìm thấy.")
    df = pd.read_csv(path)
else:
    print("Đường dẫn SAI! Python không thấy file.")

✅ Đường dẫn ĐÚNG! File đã tìm thấy.


In [ ]:
# 2. FEATURE ENGINEERING 
def feature_engineering(df):
    temp_df = df.copy()
    conditions = [
        (temp_df['bmi'] < 18.5),
        (temp_df['bmi'] < 25),
        (temp_df['bmi'] < 30),
        (temp_df['bmi'] >= 30)
    ]
    choices = ['Underweight', 'Normal', 'Overweight', 'Obese']
    temp_df['bmi_category'] = np.select(conditions, choices, default='Normal')
    
    temp_df['obese_smoker'] = ((temp_df['bmi'] >= 30) & (temp_df['smoker'] == 'yes')).astype(int)
    return temp_df

df_fe = feature_engineering(df)

In [14]:
# 3. CHIA TÁCH DỮ LIỆU
X = df_fe.drop('charges', axis=1)
y = df_fe['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:

# 4. ĐỊNH NGHĨA BỘ TIỀN XỬ LÝ (PREPROCESSOR)
def get_preprocess(X):
    categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
    numerical_cols = X.select_dtypes(include=['int64', 'float64', 'number']).columns.tolist()
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_cols),
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
        ]
    )
    return preprocessor

preprocessor = get_preprocess(X_train)

In [16]:
# 5. TIỀN XỬ LÝ VÀ TRAIN MODEL
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # Tạm thời bỏ các tham số gán cứng đi, chỉ giữ lại random_state
    ('model', GradientBoostingRegressor(random_state=42))
])

# Định nghĩa lưới tham số muốn thử nghiệm (nhớ thêm tiền tố 'model__' vì nó nằm trong Pipeline)
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__max_depth': [3, 4, 5]
}

# Khởi tạo GridSearchCV với 5-Fold CV
grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

print("⏳ Đang tìm kiếm tham số tối ưu cho Gradient Boosting...")
grid_search.fit(X_train, y_train)

# Lấy ra mô hình xịn nhất
best_pipeline = grid_search.best_estimator_
print(f"✅ Tham số tốt nhất tìm được: {grid_search.best_params_}")

⏳ Đang tìm kiếm tham số tối ưu cho Gradient Boosting...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
✅ Tham số tốt nhất tìm được: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 300}


In [17]:
# 6. ĐÁNH GIÁ MÔ HÌNH
y_pred = best_pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("="*50)
print(f"{'KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (GRADIENT BOOSTING)':^50}")
print("="*50)
print(f"R2 Score (Trên tập Test): {r2:>15.4f}")
print(f"MAE (Sai số tuyệt đối): {f'{mae:,.2f}':>22}")
print(f"RMSE (Sai số căn bậc hai): {f'{rmse:,.2f}':>20}")
print("="*50)


   KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (GRADIENT BOOSTING)   
R2 Score (Trên tập Test):          0.8683
MAE (Sai số tuyệt đối):               2,725.05
RMSE (Sai số căn bậc hai):             4,520.91


In [18]:
import wandb

run = wandb.init(
    project="Medical-Insurance-Cost-Prediction",  
    name="GradientBoostingRegressor",
    config={
        "model": "GradientBoostingRegressor",
        "C": 1.0,
        "cv_folds": 5,
        "max_iter": 1000,
        "test_size": 0.2,
        "random_state": 42
    }
)

print(run.config)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:


KeyboardInterrupt: 

In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.5, color='blue')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Đường chuẩn y=x
ax.set_xlabel('Chi phí thực tế (Actual)')
ax.set_ylabel('Chi phí dự đoán (Predicted)')
ax.set_title('Biểu đồ Thực tế vs Dự đoán - SVR')

wandb.log({
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "Actual_vs_Predicted": wandb.Image(fig)
})

plt.close(fig)